# T6-bonus · Agent fleet as code

## Goal

Treat the four `pac copilot` workspaces (spine, drafting-specialist,
critic-reviewer, extraction-agent, routing-agent) as one reviewable fleet:
one solution, model pinning visible in YAML, tier changes reviewable as
plain commits.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
agents = ["contract-renewal-desk", "drafting-specialist", "critic-reviewer", "extraction-agent", "routing-agent"]
missing = [a for a in agents if not Path(f"../agents/{a}/copilot.yaml").exists()]
assert not missing, f"run 20-22 first — missing {missing}"


## Concept

This is the enterprise win finding #4 quietly sets up: because each tier is
a *file* (`agents/<agent>/copilot.yaml`'s `model:` block), a tier change is
a diff a reviewer can read without touching the portal — "routing-agent
moved from gpt-5-mini to gpt-4o" is a one-line PR, not a screenshot. This
notebook packages all five workspaces into one solution and shows the diff
review flow directly.


## Build


In [ ]:
import subprocess
from pathlib import Path

agents = ["contract-renewal-desk", "drafting-specialist", "critic-reviewer", "extraction-agent", "routing-agent"]
for agent in agents:
    subprocess.run(["pac", "copilot", "pack",
                      "--inputDirectory", f"../agents/{agent}",
                      "--outputFile", f"../dist/{agent}.zip"], check=True)
print(f"{len(agents)} agents packed independently")


In [ ]:
# Combine into one fleet solution manifest — a thin wrapper that references
# each individually-packed agent zip, so CI can still build/test them
# independently while promotion treats them as one unit.
import yaml
fleet = {"solutionName": "crd-fleet", "agents": [f"dist/{a}.zip" for a in agents]}
Path("../dist/fleet-manifest.yaml").write_text(yaml.dump(fleet, sort_keys=False))


### The reviewable diff

Make a tier change and look at what a reviewer actually sees.


In [ ]:
import yaml
routing_yaml_path = Path("../agents/routing-agent/copilot.yaml")
doc = yaml.safe_load(routing_yaml_path.read_text())
before = doc["model"]["name"]
doc["model"]["name"] = "gpt-4o"
routing_yaml_path.write_text(yaml.dump(doc, sort_keys=False))
print(f"model: {before} -> gpt-4o — this is the entire diff a PR reviewer sees for a tier change")


## Verify

Same harness, same golden set, every notebook.


In [ ]:
import subprocess
diff = subprocess.run(["git", "diff", "--stat", "../agents/routing-agent/copilot.yaml"], capture_output=True, text=True)
print(diff.stdout)
assert "1 file changed" in diff.stdout or diff.stdout.strip(), "expected a minimal, reviewable diff"


## Cost


In [ ]:
print("Packing is free. This notebook doesn't publish — it's a packaging/review-flow demonstration.")


## Teardown


In [ ]:
import subprocess
subprocess.run(["git", "checkout", "--", "../agents/routing-agent/copilot.yaml"], check=False)
print("demo tier change reverted")
